# # 과제명 : Multi Taget Classification 모델 생성

(분석 요건)
- 분석 주제 : KOSPI 지수 & 건강식품 지수의 상승 가능성(Probability) 예측
- 분석 단위 : Day
- 문제 유형 : 분류 문제
- 종속 변수 1 : D+3일 KOSPI 지수 상승 여부 (Target_1)
- 종속 변수 2 : D+3일 건강식품 지수 상승 여부 (Target_2)
- 설명 변수 : 외부 오픈데이터를 활용한 거시경제·금융시장·소비 트렌드·기상 요인
(해외 지수, 유가, 금리, 쇼핑 지수, 날씨 등)
- 학습 기간 : 924일 (약 2년 6.5개월)
- 평가 기간 : Public 평가 - 100일, Private 평가 - 200일

(평가 기준)
- 평가 지표 : AUC (Area Under the ROC Curve)
- 평가 Score : Public Score (전체 테스트 데이터 중 50%),  Private Score (전체 테스트 데이터 중 100%)
- 최종 평가 : 정량 평가 (Private Score) + 정성 평가 (보고서, Code)

# 데이터 준비
데이터 불러오기

In [1]:
from pathlib import Path

import pandas as pd

# 프로젝트 루트 또는 notebooks/에서 실행할 수 있도록 경로 설정
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "submissions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Train data
train = pd.read_csv(DATA_DIR / "train.csv", encoding="cp949")
train["Temp"] ="train"                                          # Dataset 구분 추가
train["Target_1"] = train["Target_1"].astype("int")             # Target_1 변수 타입 변경
train["Target_2"] = train["Target_2"].astype("int")             # Target_2 변수 타입 변경

# Test data
test = pd.read_csv(DATA_DIR / "test_public.csv", encoding="cp949")  # Public Test Set 불러오기
#test = pd.read_csv(DATA_DIR / "test_private.csv", encoding="cp949") # Private Test Set 불러오기
test["Temp"] ="test"                                            # Dataset 구분 추가



/var/folders/87/mgh__9fn40s3k8wxjg7yzqfw0000gn/T/ipykernel_92667/2267693173.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["Temp"] ="train"                                          # Dataset 구분 추가
/var/folders/87/mgh__9fn40s3k8wxjg7yzqfw0000gn/T/ipykernel_92667/2267693173.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["Temp"] ="test"                                            # Dataset 구분 추가


# W&B 실험 추적
아래 셀은 실험마다 하나의 W&B Run을 만듭니다. 새 모델을 실험할 때 `EXPERIMENT_NAME`, `MODEL_NAME`, `MODEL_PARAMS`만 바꿔 다시 실행하세요.

In [2]:
import sys

import wandb

# src/tracking.py의 공통 프로젝트 설정 사용
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.tracking import init_run

EXPERIMENT_NAME = "baseline-random-forest"
MODEL_NAME = "RandomForestClassifier"
MODEL_PARAMS = {
    "n_estimators": 100,
    "max_depth": 10,
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "random_state": 42,
}

# 이 셀을 다시 실행해도 이전 Run이 열린 채 남지 않게 종료
if wandb.run is not None:
    wandb.finish()

run = init_run(
    name=EXPERIMENT_NAME,
    job_type="training",
    tags=["baseline", "random-forest", "multi-target"],
    config={
        "model/name": MODEL_NAME,
        "model/params": MODEL_PARAMS,
        "data/train_file": "train.csv",
        "data/test_file": "test_public.csv",
        "data/validation_ratio": 0.3,
        "features/daily_change": True,
        "preprocessing/numeric_missing": "median",
        "preprocessing/categorical_missing": "Unknown",
        "preprocessing/categorical_encoding": "one-hot",
    },
)
print(f"W&B Run 시작: {run.name} ({run.url})")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/pds2023/.netrc.


wandb: Currently logged in as: pds2023 (skala-autoinsight) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: setting up run k806dnrh


wandb: Tracking run with wandb version 0.30.0


wandb: Run data is saved locally in /Users/pds2023/IdeaProjects/SKALA_AutoInsight/notebooks/wandb/run-20261013_141731-k806dnrh
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run baseline-random-forest


wandb: ⭐️ View project at https://wandb.ai/skala-autoinsight/autoinsight_mini_project


wandb: 🚀 View run at https://wandb.ai/skala-autoinsight/autoinsight_mini_project/runs/k806dnrh


W&B Run 시작: baseline-random-forest (https://wandb.ai/skala-autoinsight/autoinsight_mini_project/runs/k806dnrh)


데이터 병합하기

In [3]:
df = pd.concat([train, test], axis=0, ignore_index=True)
df = df.sort_values(by='Time_step', ascending=True).reset_index(drop=True)
df

,Time_step,Target_1,Target_2,Weekday,FI_Nasdaq,FI_Dow,FI_Sp500,FI_Brent,FI_WTI,FI_Dubai,...,WE_평균기온,WE_최고기온,WE_최저기온,WE_평균풍속_m_s_,WE_최대풍속_m_s_,WE_최대풍속풍향_deg_,WE_최대순간풍속_m_s_,WE_최대순간풍속풍향_deg_,WE_강수량,Temp
0,1,0.0,0.0,3,0.030456,0.307678,0.188059,0.484618,0.761049,0.481937,...,0.894845,0.868787,0.871102,0.240741,0.362745,0.794118,0.188525,0.794118,0.016064,train
1,2,0.0,0.0,4,0.030404,0.312452,0.189019,0.493280,0.764819,0.491314,...,0.923711,0.868787,0.923077,0.240741,0.294118,0.411765,0.241803,0.470588,0.016064,train
2,3,1.0,0.0,5,0.026407,0.313351,0.184762,0.488053,0.759909,0.485319,...,0.927835,0.886680,0.935551,0.333333,0.362745,0.794118,0.172131,0.794118,0.016064,train
3,4,1.0,1.0,6,0.028417,0.319438,0.188437,0.494176,0.764732,0.492237,...,0.929897,0.914513,0.908524,0.240741,0.333333,0.794118,0.159836,0.794118,0.016064,train
4,5,0.0,1.0,7,0.028417,0.319438,0.188437,0.494176,0.764732,0.492237,...,0.952577,0.904573,0.943867,0.314815,0.235294,0.205882,0.135246,0.264706,0.000000,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1019,1020,NaN,NaN,7,0.505440,0.464756,0.492913,0.196685,0.588039,0.164950,...,0.649485,0.616302,0.681913,0.462963,0.401961,0.735294,0.209016,0.794118,0.020080,test
1020,1021,NaN,NaN,1,0.505440,0.464756,0.492913,0.196685,0.588039,0.164950,...,0.698969,0.707753,0.673597,0.407407,0.450980,0.941176,0.282787,0.941176,0.000000,test
1021,1022,NaN,NaN,2,0.544897,0.547967,0.563910,0.231183,0.608997,0.176480,...,0.698969,0.745527,0.644491,0.500000,0.382353,0.882353,0.258197,0.882353,0.239357,test
1022,1023,NaN,NaN,3,0.535989,0.512335,0.539536,0.228793,0.610224,0.174481,...,0.573196,0.552684,0.580042,0.240741,0.235294,0.794118,0.127049,0.794118,0.152610,test


# 파생변수 생성

In [4]:
# === 전일 대비 증감율

import numpy as np

# 증감율 계산 대상 변수
cols = df.columns.difference(['Time_step', 'Target_1', 'Target_2', 'Weekday', 'Temp'])

# 증감율 변수 생성
df_rate = (df[cols] - df[cols].shift(1)) / df[cols].shift(1).replace(0, np.nan)

# 컬럼 이름 변경
df_rate = df_rate.add_suffix('_chg')

# 원본과 병합
df = pd.concat([df, df_rate], axis=1)

# 첫 번째 행은 전일값 없음으로 NaN → 제거
df = df.iloc[1:].reset_index(drop=True)

# Data Partition
 - 학습용과 평가용 데이타 우선 분리
 - 학습용 데이타는 Train vs Valid Set 으로 분리
 - !! Timeseries Split 주의 : Shuffle 사용 금지 (시간 순서 유지 필수)

In [5]:
# Train 과 Test 분리
df_train = df.query("Temp == 'train'")
df_test  = df.query("Temp == 'test'")

# Train 과 Validation 분리
split_idx = int(len(df_train) * 0.7)

# 시계열 기준 분리
df_valid = df_train.iloc[split_idx:].reset_index(drop=True)
df_train = df_train.iloc[:split_idx].reset_index(drop=True)
print(df_train.shape)
print(df_valid.shape)
print(df_test.shape)

(646, 571)
(277, 571)
(100, 571)


# 분석 변수 선택


원래 데이터 복제

In [6]:
df_train_raw = df_train.copy()
df_valid_raw  = df_valid.copy()
df_test_raw  = df_test.copy()

분석 변수 선택

In [7]:
df_train = df_train.drop(columns=["Time_step", "Temp"])
df_valid = df_valid.drop(columns=["Time_step", "Temp"])
df_test = df_test.drop(columns=["Time_step", "Temp"])

# X, Y 분리

In [8]:
# Train
X_train = df_train.drop(columns=["Target_1", "Target_2"])
Y_train = df_train[['Target_1', 'Target_2']]

# Valid
X_valid = df_valid.drop(columns=["Target_1", "Target_2"])
Y_valid = df_valid[['Target_1', 'Target_2']]

# Test
X_test = df_test.drop(columns=["Target_1", "Target_2"])
Y_test = df_test[['Target_1', 'Target_2']]

# 단일값 변수 처리

In [9]:
# 고유값 개수 계산 (NaN 포함)
unique_counts = X_train.nunique(dropna=False)

# 고유값이 1개 이하인 컬럼 식별
cols_to_drop = unique_counts[unique_counts <= 1].index.tolist()

# 데이터셋에서 삭제
X_train = X_train.drop(columns=cols_to_drop)
X_valid = X_valid.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f"삭제된 컬럼: {cols_to_drop}")

삭제된 컬럼: ['KE_QR체크인', 'KE_QR체크인_chg']


# Missing 처리
Missing 데이터 확인

In [10]:
df_train.isnull().sum()

Target_1              0
Target_2              0
Weekday               0
FI_Nasdaq             0
FI_Dow                0
                     ..
WE_최대풍속_m_s__chg      2
WE_최대풍속풍향_deg__chg    3
WE_최저기온_chg           1
WE_평균기온_chg           1
WE_평균풍속_m_s__chg      0
Length: 569, dtype: int64

Missing 대체 (연속형)
 - median 대체

In [11]:
import numpy as np
import pandas as pd

# 연속형 변수 컬럼 자동 감지
continuous_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# X_train에서 중앙값 계산
medians = X_train[continuous_cols].median()

# Missing 값을 중앙값으로 대체
X_train[continuous_cols] = X_train[continuous_cols].fillna(medians)
X_valid[continuous_cols] = X_valid[continuous_cols].fillna(medians)
X_test[continuous_cols] = X_test[continuous_cols].fillna(medians)

Missing 대체 (범주형)
 - New class ('Unknown') 대체

In [12]:
# 신규 범주 (Unknown) 로 대체
import pandas as pd

# 범주형 컬럼 선택 (object + category)
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    # category 타입이면 Unknown 먼저 추가
    if X_train[col].dtype.name == 'category':
        X_train[col] = X_train[col].cat.add_categories('Unknown')
        X_valid[col] = X_valid[col].cat.add_categories('Unknown')
        X_test[col] = X_test[col].cat.add_categories('Unknown')

    # Missing → 'Unknown'
    X_train[col] = X_train[col].fillna('Unknown')
    X_valid[col] = X_valid[col].fillna('Unknown')
    X_test[col] = X_test[col].fillna('Unknown')

# 학습 준비

One-Hot Encoding

In [13]:
import pandas as pd

# 범주형 컬럼 선택 (Train 기준)
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# One-Hot Encoding
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=False, dummy_na=False)
X_valid = pd.get_dummies(X_valid, columns=cat_cols, drop_first=False, dummy_na=False)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=False, dummy_na=False)

# Validation/Test를 Train의 컬럼 구조에 맞추기
# - Train에 있고 Validation/Test에 없는 컬럼 → 0으로 채움
# - Train에 없고 Validation/Test에 있는 컬럼 → 제거
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test  shape:", X_test.shape)

X_train shape: (646, 565)
X_valid shape: (277, 565)
X_test  shape: (100, 565)


# RandomForest
 학습 -> 예측 -> 평가
  - API 문서 : https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Multi Target 처리 : Target별 학습
models = {}
for target in Y_train.columns:
    models[target] = RandomForestClassifier(**MODEL_PARAMS)
    models[target].fit(X_train, Y_train[target])

# 예측 : '1의 확률' 추출
prob_df_valid = pd.DataFrame({target: models[target].predict_proba(X_valid)[:, 1] for target in Y_train.columns})  # Valid Set
prob_df_test = pd.DataFrame({target: models[target].predict_proba(X_test)[:, 1] for target in Y_train.columns})    # Test Set

# Valid set 평가
print("\n=== [Valid Set : Target별 AUC 점수] ===")
auc_list = []
for target in Y_train.columns:
    auc = roc_auc_score(Y_valid[target], prob_df_valid[target])
    auc_list.append(auc)
    print(f"{target:6s} ==> AUC: {auc:.4f}")
mean_auc = float(np.mean(auc_list))
print(f"평균     ==> AUC: {mean_auc:.4f}")

# 베이스라인 성능과 데이터 규모를 현재 W&B Run에 기록
metrics = {
    **{f"valid/{target}_auc": float(auc) for target, auc in zip(Y_train.columns, auc_list)},
    "valid/mean_auc": mean_auc,
}
run.config.update({
    "data/train_rows": len(X_train),
    "data/valid_rows": len(X_valid),
    "data/test_rows": len(X_test),
    "data/feature_count": X_train.shape[1],
    "features/dropped_constant_columns": cols_to_drop,
}, allow_val_change=True)
run.log(metrics)
run.summary.update(metrics)


=== [Valid Set : Target별 AUC 점수] ===
Target_1 ==> AUC: 0.5521
Target_2 ==> AUC: 0.9452
평균     ==> AUC: 0.7486


- Test 결과 제출 파일 생성

In [15]:
# Test Set 예측 결과
df_test_raw["pred_1"] = prob_df_test["Target_1"].values
df_test_raw["pred_2"] = prob_df_test["Target_2"].values
submission = df_test_raw[["Time_step", "pred_1", "pred_2"]]
submission_path = OUTPUT_DIR / 'public_1반_박다솔_1.csv'
submission.to_csv(submission_path, sep=',', na_rep='NaN', index=False)    # Public Score 용, 홍길동_1: 제출자명_제출회차
#submission.to_csv(OUTPUT_DIR / 'private_1반_박다솔.csv', sep=',', na_rep='NaN', index=False)   # Private Score 용, 홍길동: 제출자명

## W&B 결과 저장 및 Run 종료
제출 파일과 검증 예측값을 저장하고 Run을 정상 종료합니다. 실험 하나가 끝날 때마다 이 셀을 실행하세요.

In [16]:
# 검증 예측값을 Table로 남겨 Run 페이지에서 실험 간 비교 가능
valid_predictions = pd.DataFrame({
    "Target_1_actual": Y_valid["Target_1"].reset_index(drop=True),
    "Target_1_probability": prob_df_valid["Target_1"],
    "Target_2_actual": Y_valid["Target_2"].reset_index(drop=True),
    "Target_2_probability": prob_df_valid["Target_2"],
})
run.log({"valid/predictions": wandb.Table(dataframe=valid_predictions)})

submission_artifact = wandb.Artifact(
    name=f"{run.id}-submission",
    type="submission",
    metadata={"experiment": EXPERIMENT_NAME, "mean_auc": mean_auc},
)
submission_artifact.add_file(str(submission_path))
run.log_artifact(submission_artifact)

run.finish()
print("W&B 기록 완료")

wandb: updating run metadata; uploading artifact run-k806dnrh-validpredictions-0Uqj3Q; uploading artifact k806dnrh-submission


wandb: uploading artifact run-k806dnrh-validpredictions-0Uqj3Q; uploading artifact k806dnrh-submission


wandb: uploading requirements.txt; uploading media/table/valid/predictions_1_c10e29e8c7af387e5ee6.table.json; uploading output.log; uploading wandb-summary.json; uploading config.yaml (+ 1 more)


wandb: uploading history steps 0-1, summary, console lines 0-0


wandb: 
wandb: Run history:
wandb: valid/Target_1_auc ▁
wandb: valid/Target_2_auc ▁
wandb:     valid/mean_auc ▁
wandb: 
wandb: Run summary:
wandb: valid/Target_1_auc 0.55212
wandb: valid/Target_2_auc 0.94517
wandb:     valid/mean_auc 0.74865
wandb: 


wandb: 🚀 View run baseline-random-forest at: https://wandb.ai/skala-autoinsight/autoinsight_mini_project/runs/k806dnrh
wandb: ⭐️ View project at: https://wandb.ai/skala-autoinsight/autoinsight_mini_project
wandb: Synced 5 W&B file(s), 1 media file(s), 4 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261013_141731-k806dnrh/logs


W&B 기록 완료
